# Growing Neural Cellular Automata — gecko baseline

Reproduction of [Mordvintsev et al., *Growing Neural Cellular Automata*, Distill 2020](https://distill.pub/2020/growing-ca/) in PyTorch.

**Runtime → Change runtime type → GPU** before running. The library code lives in `diffluoditty/`; this notebook just drives it.
Training is interruptible: stop the training cell at any time, poke around, and re-run it to continue.

In [ ]:
# Setup: clone the repo (Colab) or use the local checkout.
import os
if os.path.exists("../diffluoditty/__init__.py"):   # running from notebooks/ in a local checkout
    %cd ..
elif not os.path.exists("diffluoditty/__init__.py"):
    if not os.path.exists("diffluoditty"):
        !git clone -q https://github.com/aphid91/diffluoditty.git
    %cd diffluoditty
!git pull -q

# Re-import edited library code automatically, so you can hack on the .py files mid-session.
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML

from diffluoditty import NCA, Trainer, experiment_config, load_emoji, pad_target, circle_masks, half_mask
from diffluoditty import viz

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

## Target
The 🦎 emoji, shrunk to 40px and padded by 16px on each side → 72×72, with premultiplied alpha.
`load_image(path)` works for any RGBA PNG.

In [ ]:
target = pad_target(load_emoji("gecko", max_size=40), padding=16)
print(target.shape)
viz.show_images(viz.to_rgb(target)[None], scale=3)

## Configuration
The three experiments from the paper differ only in two settings:

| experiment | `use_pool` | `damage_n` | learns to… |
|---|---|---|---|
| `growing` | False | 0 | grow from a seed (but tends to explode or decay afterwards) |
| `persistent` | True | 0 | grow, and then *stay* that shape |
| `regenerating` | True | 3 | grow, persist, and repair damage |

Everything else defaults to the paper: batch 8, pool 1024, 64–96 steps per rollout, Adam at 2e-3 → 2e-4 at step 2000, per-parameter gradient normalisation, 8000 iterations.
Pass overrides as keyword arguments, e.g. `experiment_config("regenerating", batch_size=16, grad_checkpoint_every=16)`.

In [ ]:
EXPERIMENT = "regenerating"
N_ITERS = 8000

cfg = experiment_config(EXPERIMENT)
cfg

In [ ]:
torch.manual_seed(0)
nca = NCA(size=tuple(target.shape[-2:]), channels=16, hidden=128, fire_rate=0.5).to(device)
trainer = Trainer(nca, target, cfg)
print(f"{sum(p.numel() for p in nca.parameters())} parameters")

## Train
The plot shows log10 loss, and the last batch before and after its rollout. In `regenerating` mode the first sample in each batch is a fresh seed and the last three are damaged.
Interrupt whenever you like; re-running continues where it stopped.

In [ ]:
try:
    trainer.train(N_ITERS - trainer.step_count, callback=viz.show_progress, callback_every=100)
except KeyboardInterrupt:
    print(f"interrupted at step {trainer.step_count}")

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
trainer.save(f"checkpoints/{EXPERIMENT}.pt")
# Later / in a fresh session (rebuild nca + trainer with the same settings first):
# trainer.load(f"checkpoints/{EXPERIMENT}.pt")

## Training diagnostics

In [ ]:
h = trainer.history
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
viz.plot_history(h, axes[0])
axes[1].semilogy([r["grad_norm"] for r in h], ".", markersize=1, alpha=0.3); axes[1].set_title("raw gradient norm (before normalisation)")
axes[2].plot([r["lr"] for r in h]); axes[2].set_title("learning rate")
plt.show()

A random sample of the pool. After training, these should all look like (possibly slightly damaged) geckos.

In [ ]:
if trainer.pool is not None:
    idx = torch.randperm(cfg.pool_size)[:32]
    viz.show_images(viz.to_rgb(nca.render(trainer.pool.state[idx])), ncols=8)

## Growth from a single seed

In [ ]:
frames = viz.record(nca, nca.seed(1), n_steps=300, every=2)
HTML(viz.animate(frames, scale=3).to_jshtml())

## Regeneration
Grow for 200 steps, cut four copies in different ways, then watch them heal. (Only the `regenerating` model is expected to do this well.)

In [ ]:
h, w = target.shape[-2:]
cuts = torch.cat([
    half_mask(1, h, w, "right"),
    half_mask(1, h, w, "top"),
    circle_masks(2, h, w),
]).to(device)

frames = viz.record(nca, nca.seed(4), n_steps=500, every=2, events={200: lambda s: nca.damage(s, cuts)})
HTML(viz.animate(frames, scale=2).to_jshtml())

## Long-term stability
Does the pattern hold its shape far beyond the 96 steps it was trained on?

In [ ]:
with torch.no_grad():
    x, snaps, times = nca.seed(1), [], [100, 500, 1000, 2000, 5000]
    for t in range(1, times[-1] + 1):
        x = nca.step(x)
        if t in times:
            snaps.append(viz.to_rgb(nca.render(x))[0])
viz.show_images(snaps, titles=[f"t={t}" for t in times], scale=2.5)

Save an animation to the Colab session (Files panel on the left): `viz.animate(frames).save("regen.gif", writer="pillow")` or `.save("regen.mp4")`.